In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Data Preprocessing

In [ ]:
data = pd.read_csv("all_participants_data.csv", index_col="Participant")

indexes = data.index

#getting list of median arousals and valences

allMedianArousals = list(data["median_arousal"])
allMedianValences = list(data["median_valence"])

#getting median of the rows

medianArousal = data["median_arousal"].median()
medianValence = data["median_valence"].median()

data = data.drop(columns=['median_arousal', 'median_valence'])

scaler = StandardScaler()
scaledArray = scaler.fit_transform(data)
scaledData = pd.DataFrame(scaledArray, index=indexes)

print(scaledData)


In [ ]:
#getting lists of low and high arousals and valences

lowArousal = [(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] <= medianArousal]
highArousal = [(row, 1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > medianArousal]

lowValence = [(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] <= medianValence]
highValence = [(row, 1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > medianValence]

#getting percentile classes for arousals and valences

arousalPercentiles = np.percentile(allMedianArousals, [20, 40, 60, 80])
arousalPercentileClasses = []

arousalPercentileClasses.append([(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] <= arousalPercentiles[0]])
for i in range(0, 3):
    arousalPercentileClasses.append([(row, i+1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > arousalPercentiles[i] and allMedianArousals[j] <= arousalPercentiles[i+1]])
arousalPercentileClasses.append([(row, 4, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > arousalPercentiles[3]])

arousalPercentileClasses = sum(arousalPercentileClasses, [])


valencePercentiles = np.percentile(allMedianValences, [20, 40, 60, 80])
valencePercentileClasses = []

valencePercentileClasses.append([(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] <= valencePercentiles[0]])
for i in range(0, 3):
    valencePercentileClasses.append([(row, i+1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > valencePercentiles[i] and allMedianValences[j] <= valencePercentiles[i+1]])
valencePercentileClasses.append([(row, 4, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > valencePercentiles[3]])

valencePercentileClasses = sum(valencePercentileClasses, [])

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression(max_iter=10000)

# Neural Network Classifier

In [ ]:
from sklearn.neural_network import MLPClassifier

model2 = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=1000, random_state=42)

# LSTM Classifier

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=288,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)     
        last_out = lstm_out[:, -1, :]  
        out = self.fc(last_out)        
        return out
    
model3 = LSTMClassifier(1, 16, 2, 2)

In [ ]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

# **Model Testing**

# Multiclass classification

In [ ]:
# testing model 1

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in valencePercentileClasses]
Y = [element[1] for element in valencePercentileClasses]
orderedIndexes = [element[2] for element in valencePercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model1.fit(Xtrain, Ytrain)

    Ypred = model1.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

In [ ]:
# testing model 2

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in valencePercentileClasses]
Y = [element[1] for element in valencePercentileClasses]
orderedIndexes = [element[2] for element in valencePercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model2.fit(Xtrain, Ytrain)

    Ypred = model2.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

In [ ]:
# testing model 3

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in arousalPercentileClasses]
Y = [element[1] for element in arousalPercentileClasses]
orderedIndexes = [element[2] for element in arousalPercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    Xtrain, Ytrain = torch.tensor(Xtrain, dtype = torch.float).unsqueeze(1), torch.tensor(Ytrain, dtype = torch.long)
    Xtest, Ytest = torch.tensor(Xtest, dtype = torch.float).unsqueeze(1), torch.tensor(Ytest, dtype = torch.float).unsqueeze(1)

    optimizer = optim.Adam(model3.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model3(Xtrain)
        loss = criterion(outputs, Ytrain)
        loss.backward()
        optimizer.step()

    model3.eval()
    with torch.no_grad():
        probs = torch.softmax(model3(Xtest), dim=1)
        expected = np.dot(probs, np.arange(5))

        probs = probs.argmax(axis=1)




        results.append((pearsonr(np.array(probs).ravel(), np.array(expected).ravel())[0], CCcoefficient(probs, expected)))

print(results)
    

# Binary classification

In [ ]:
# testing model 1

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in lowValence + highValence]
Y = [element[1] for element in lowValence + highValence]
orderedIndexes = [element[2] for element in lowValence + highValence]
results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model1.fit(Xtrain, Ytrain)

    Ypred = model1.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

In [ ]:
# testing model 2

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in lowArousal + highArousal]
Y = [element[1] for element in lowArousal + highArousal]
orderedIndexes = [element[2] for element in lowArousal + highArousal]
results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model2.fit(Xtrain, Ytrain)

    Ypred = model2.predict_proba(Xtest)
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

In [ ]:
# testing model 3

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in lowValence + highValence]
Y = [element[1] for element in lowValence + highValence]
orderedIndexes = [element[2] for element in lowValence + highValence]
results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    Xtrain, Ytrain = torch.tensor(Xtrain, dtype = torch.float).unsqueeze(1), torch.tensor(Ytrain, dtype = torch.long)
    Xtest, Ytest = torch.tensor(Xtest, dtype = torch.float).unsqueeze(1), torch.tensor(Ytest, dtype = torch.float).unsqueeze(1)

    optimizer = optim.Adam(model3.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model3(Xtrain)
        loss = criterion(outputs, Ytrain)
        loss.backward()
        optimizer.step()

    model3.eval()
    with torch.no_grad():
        probs = torch.softmax(model3(Xtest), dim=1)
        expected = np.dot(probs, np.arange(2))

        probs = probs.argmax(axis=1)




        results.append((pearsonr(np.array(probs).ravel(), np.array(expected).ravel())[0], CCcoefficient(probs, expected)))

print(results)
    